# Clusters of Galaxies Summary Statistics DEMO

> An executed version of this notebook can be found [here](https://me.lsst.eu/maguena/euclid/cloecl/DEMO_Clusters_of_Galaxies_Summary_Statistics.html)
> (user:cloecl, pwd:cluster)

In [ ]:
import time

import numpy as np

from cloelib.cosmology.camb_cosmology import CAMBBackground, CAMBLinearPerturbations
from cloelib.observables.clusters.clustering import HaloClustering
from cloelib.observables.clusters.covariance import HaloCovariance
from cloelib.observables.clusters.halo_statistics import HaloStatistics
from cloelib.observables.clusters.hmf_bias import CastroHMFBias
from cloelib.observables.clusters.profile import ProfileNFW
from cloelib.observables.clusters.selection_function import SelectionFunction
from cloelib.summary_statistics.clusters import (
    ClusterClustering,
    ClusterCounts,
    ClusterStatisticsModeling,
    ClusterWeakLensing,
)

## Instanciate Cosmology

In [ ]:
# Cosmology parameters
t0 = time.time()
_H0 = 67.0
_h = _H0 / 100.0
_omch2 = 0.12
_ombh2 = 0.022
_cosmo_pars = dict(
    H0=_H0,
    Omega_cdm0=_omch2 / _h**2,
    Omega_b0=_ombh2 / _h**2,
    Omega_k0=0.0,
    w0=-1.0,
    wa=0.0,
    ns=0.96,
    mnu=0.06,
    As=2e-9,
    gamma_MG=0.0,
)

background = CAMBBackground(**_cosmo_pars)
perturbations = CAMBLinearPerturbations(background, np.linspace(0.0, 2.0, 100))

_cosmo_pars_fid = {**_cosmo_pars}
_cosmo_pars_fid["H0"] = 73.0
background_fid = CAMBBackground(**_cosmo_pars_fid)
perturbations_fid = CAMBLinearPerturbations(
    background_fid, np.linspace(0.0, 2.0, 100)
)
print(f"cosmo     :  {time.time()-t0:.4f} seconds")

## Instanciate Cluster observables

In [ ]:
# Parameters
_sel_pars = dict(
    A_l=52.0,
    B_l=0.9,
    C_l=0.5,
    sig_A_l=0.2,
    sig_B_l=-0.05,
    sig_C_l=0.001,
    sig_lambda_norm=0.9,
    sig_lambda_z=0.1,
    sig_lambda_exponent=0.4,
    sig_z_z=0.025,
    sig_z_lambda=5.0e-6,
)

_prof_pars = dict(
    r_interp=np.logspace(-10, 2.5, 200),
    two_halo="None",
    offcentering=False,
    rms_off=0.0,
    f_off=0.0,
    trunc_fact=3.0,
    zs_max=2.0,
    mean_nz=0.4,
    sigma_nz=0.3,
    alpha_nz=0.4,
)

integ_k_arr = np.geomspace(1e-4, 10, 500)
integ_mass_arr = np.logspace(12.0, 16.0, 51)
integ_lambda_true_arr = np.geomspace(5.0, 250.0, 51)
integ_ztrue_arr = np.linspace(1.0e-5, 6.0 - 1.0e-5, 200)

halo_concentration = 0.1
overdensity_type = "vir"
area = 10313

# Integration bins

z_obs_nc_edges = np.linspace(0.2, 1.8, 9)
lambda_obs_nc_edges = np.array([20.0, 30.0, 45.0, 60.0, 500.0])
z_obs_profile_edges = np.linspace(0.2, 1.8, 9)
lambda_obs_profile_edges = np.array([20.0, 30.0, 45.0, 60.0, 500.0])
radius_profile_edges = np.linspace(5.0, 100.0, 11)
lambda_obs_clustering_edges = np.array([20, 30, 500])
radius_clustering_edges = np.geomspace(20.0, 130.0, 31)
zed_obs_clustering_edges = np.arange(0.2, 1.81, 0.4)

In [ ]:
# Istanciate objects
t0 = time.time()
selectionFunction = SelectionFunction(**_sel_pars)
HSCastro = CastroHMFBias(
    halo_statistics=HaloStatistics(
        perturbations,
        z=integ_ztrue_arr,
        k=integ_k_arr,
        overdensity_type=overdensity_type,
    )
)
covariance = HaloCovariance(
    perturbations, area=area, nbins_zob=len(z_obs_nc_edges), k=integ_k_arr
)
profileNFW = ProfileNFW(HSCastro, k=integ_k_arr, z=integ_ztrue_arr, **_prof_pars)
haloClustering = HaloClustering(
    perturbations, perturbations_fid, selectionFunction, k=integ_k_arr
)

print(f"init obs  :  {time.time()-t0:.4f} seconds")

## Inctanciate Cluster Summary Statistics

In [ ]:
t0 = time.time()
# Istanciate objects
cluster_statitstics_modeling = ClusterStatisticsModeling(
    HSCastro,
    selectionFunction,
    integ_k_arr=integ_k_arr,
    integ_mass_arr=integ_mass_arr,
    integ_lambda_true_arr=integ_lambda_true_arr,
    integ_ztrue_arr=integ_ztrue_arr,
    area=area,
)
cluster_counts_statistics = ClusterCounts(
    cluster_statitstics_modeling,
    covariance,
    photoz_rsd_correction=haloClustering.photoz_rsd_correction,
)
cluster_wl_statistics = ClusterWeakLensing(
    cluster_statitstics_modeling,
    profileNFW,
    halo_concentration=halo_concentration,
)
cluster_clustering_statistics = ClusterClustering(
    cluster_statitstics_modeling,
    haloClustering,
)

print(f"init stat :  {time.time()-t0:.4f} seconds")

## Compute cluster probes values

In [ ]:
t0 = time.time()
cluster_counts, counts_intermediate_integration_products = (
    cluster_counts_statistics.get_NC(
        z_obs_edges=z_obs_nc_edges,
        lambda_obs_edges=lambda_obs_nc_edges,
    )
)
print(f"nc        :  {time.time()-t0:.4f} seconds")
t0 = time.time()
cov_cluster_counts = cluster_counts_statistics.get_NC_covariance(
    z_obs_nc_edges,
    cluster_counts,
    counts_intermediate_integration_products["window_lambda_obs"],
    counts_intermediate_integration_products["window_z_obs"],
)
print(f"nc_cov    :  {time.time()-t0:.4f} seconds")
t0 = time.time()
deltasigma_mean_values = cluster_wl_statistics.get_DeltaSigma(
    z_obs_edges=z_obs_profile_edges,
    lambda_obs_edges=lambda_obs_profile_edges,
    radius_edges=radius_profile_edges,
)
print(f"dsig      :  {time.time()-t0:.4f} seconds")
t0 = time.time()
cluster_clustering, clustering_intermediate_integration_products = (
    cluster_clustering_statistics.get_xi(
        lambda_obs_edges=lambda_obs_clustering_edges,
        radius_edges=radius_clustering_edges,
        z_obs_edges=zed_obs_clustering_edges,
    )
)
print(f"xi        :  {time.time()-t0:.4f} seconds")
t0 = time.time()
cov_cluster_clustering = cluster_clustering_statistics.get_xi_covariance(
    clustering_intermediate_integration_products["pk_mean_values"],
    clustering_intermediate_integration_products["radial_shell_window"],
    clustering_intermediate_integration_products["radial_shell_volume"],
    clustering_intermediate_integration_products["window_z_obs"],
    clustering_intermediate_integration_products["cluster_counts"],
)
print(f"xi_cov    :  {time.time()-t0:.4f} seconds")